# EcoRuta Coyhaique — Instalación de dependencias

Este notebook deja todo listo para ejecutar **EcoRuta**. Realiza dos pasos:

1. **Instala la librería de Python** que necesita el servidor: **Flask**.
2. **Descarga las librerías del mapa** (Leaflet, Font Awesome y las tipografías) a una carpeta local `static/vendor/`, para que la aplicación funcione aunque la red bloquee los CDN de internet.

> ▶️ **Ejecuta este notebook una sola vez**, desde la **misma carpeta** donde está `DashEcoRuta.ipynb`. Luego abre y ejecuta `DashEcoRuta.ipynb`.

> ⚠️ **Nota importante:** el *fondo del mapa* (las imágenes de las calles, llamadas *tiles*) **siempre se descarga en vivo desde internet**. Estas descargas locales cubren la librería del mapa, los iconos y las fuentes, pero **el mapa de fondo necesita conexión** para mostrarse.

## Paso 1 · Instalar Flask

Instala el servidor web **Flask** y deja registrada la dependencia en `requirements.txt`.

In [ ]:
import sys, subprocess

# Instala Flask (servidor web local que ejecuta el dashboard)
subprocess.run([sys.executable, "-m", "pip", "install", "flask"])

# Registra la dependencia para el repositorio
with open("requirements.txt", "w", encoding="utf-8") as f:
    f.write("Flask>=3.0\n")

print("\n✔ Flask instalado y requirements.txt creado.")

## Paso 2 · Descargar las librerías del mapa (modo recomendado)

Descarga **Leaflet**, **Font Awesome** y las **tipografías** a la carpeta `static/vendor/`.
Cada descarga es independiente: si alguna falla, la aplicación usará automáticamente el CDN de internet como respaldo.

In [ ]:
import os, re, urllib.request

# User-Agent de navegador para que Google Fonts entregue archivos .woff2 modernos
UA = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
                    "(KHTML, like Gecko) Chrome/120.0 Safari/537.36"}
BASE = os.path.join("static", "vendor")

def carpeta(*partes):
    ruta = os.path.join(BASE, *partes)
    os.makedirs(ruta, exist_ok=True)
    return ruta

def descargar(url, destino, como_texto=False):
    """Descarga un archivo. Devuelve los datos si tuvo éxito, o None si falló."""
    try:
        req = urllib.request.Request(url, headers=UA)
        with urllib.request.urlopen(req, timeout=30) as r:
            data = r.read()
        with open(destino, "wb") as f:
            f.write(data)
        print("   ✔", os.path.relpath(destino))
        return data.decode("utf-8") if como_texto else data
    except Exception as e:
        print("   ✗", url.split("/")[-1], "->", e)
        return None

principales_ok = 0

# ----- Leaflet 1.9.4 (librería del mapa) -----
print("Leaflet:")
d = carpeta("leaflet")
if descargar("https://unpkg.com/leaflet@1.9.4/dist/leaflet.css", os.path.join(d, "leaflet.css")): principales_ok += 1
if descargar("https://unpkg.com/leaflet@1.9.4/dist/leaflet.js",  os.path.join(d, "leaflet.js")):  principales_ok += 1
dimg = carpeta("leaflet", "images")
for img in ["layers.png", "layers-2x.png", "marker-icon.png", "marker-icon-2x.png", "marker-shadow.png"]:
    descargar("https://unpkg.com/leaflet@1.9.4/dist/images/" + img, os.path.join(dimg, img))

# ----- Font Awesome 6.4.0 (iconos) -----
print("Font Awesome:")
dcss = carpeta("fontawesome", "css")
if descargar("https://cdnjs.cloudflare.com/ajax/libs/font-awesome/6.4.0/css/all.min.css",
             os.path.join(dcss, "all.min.css")): principales_ok += 1
dwf = carpeta("fontawesome", "webfonts")
for wf in ["fa-solid-900.woff2", "fa-solid-900.ttf", "fa-regular-400.woff2",
           "fa-regular-400.ttf", "fa-brands-400.woff2", "fa-brands-400.ttf"]:
    descargar("https://cdnjs.cloudflare.com/ajax/libs/font-awesome/6.4.0/webfonts/" + wf,
              os.path.join(dwf, wf))

# ----- Tipografías: Baloo 2 + Montserrat (Google Fonts) -----
print("Tipografías (Baloo 2 + Montserrat):")
dfonts = carpeta("fonts")
css_url = ("https://fonts.googleapis.com/css2?family=Baloo+2:wght@600;700;800"
           "&family=Montserrat:wght@400;500;600;700;800&display=swap")
css = descargar(css_url, os.path.join(dfonts, "_tmp.css"), como_texto=True)
if css:
    urls = list(dict.fromkeys(re.findall(r"https://fonts\.gstatic\.com/[^)]+?\.woff2", css)))
    descargados = 0
    for i, u in enumerate(urls):
        nombre = "font_%d.woff2" % i
        if descargar(u, os.path.join(dfonts, nombre)):
            css = css.replace(u, nombre)   # reescribe el CSS a ruta local
            descargados += 1
    with open(os.path.join(dfonts, "fonts.css"), "w", encoding="utf-8") as f:
        f.write(css)
    try:
        os.remove(os.path.join(dfonts, "_tmp.css"))
    except OSError:
        pass
    if descargados:
        principales_ok += 1
    print("   ✔ %d archivos de fuente + fonts.css" % descargados)
else:
    print("   ✗ No se pudo obtener el CSS de fuentes (se usará el CDN como respaldo).")

print("\n" + "=" * 50)
print("✔ Descarga finalizada. Componentes principales: %d/3" % principales_ok)
print("   (Leaflet, Font Awesome y Tipografías)")
print("=" * 50)

## ✅ Listo

Las librerías quedaron en `static/vendor/`. Cuando ejecutes `DashEcoRuta.ipynb`, la aplicación las usará automáticamente.

**Recordatorio:** el fondo del mapa (los *tiles*) se sigue descargando en vivo, así que para mostrar el mapa completo necesitas internet. Para una presentación, abre la app una vez con conexión para dejar el mapa en caché.

In [ ]:
import os

ruta = os.path.join("static", "vendor")
if os.path.isdir(ruta):
    print("Contenido de", ruta, ":\n")
    total = 0
    for raiz, _dirs, files in os.walk(ruta):
        for fn in sorted(files):
            f = os.path.join(raiz, fn)
            kb = max(1, os.path.getsize(f) // 1024)
            print("  %-46s %5d KB" % (f, kb))
            total += 1
    print("\nTotal de archivos:", total)
    print("\nTodo listo. Abre y ejecuta DashEcoRuta.ipynb para iniciar el dashboard.")
else:
    print("Aún no existe static/vendor. Ejecuta primero la celda del Paso 2.")